# Phase 02 — Cross-Generator Generalization and Robustness

**Research task:** Detect real vs synthetic images after training on ProGAN, then evaluate performance across eight AIGIBench test subsets with JPEG compression, Gaussian noise, and downsampling/upsampling.

**Recorded results:** Best in-domain validation accuracy 99.93%; average original-image accuracy across the eight selected test subsets 60.3%; average across four conditions 53.7%.

**Before running:** This is a Kaggle-oriented, compute-intensive training pipeline. Review data licenses, disk requirements, and `TMP_DIR`/`WORK_DIR`. A dangerous broad temporary-directory cleanup function from the uploaded notebook was replaced with a scoped project-cache cleanup; model and evaluation logic is unchanged. All output has been moved to `../results/phase-02/`.

Full project details: [Phase 02 README](README.md).

## 1. Imports, configuration, downloads, and helper functions

In [ ]:
import os, io, json, zipfile, shutil, random, traceback, requests
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE   = 224
BATCH_SIZE = 64
EPOCHS     = 30
SEED       = 42
TMP_DIR    = "/kaggle/tmp/AIGIBench"
WORK_DIR   = "/kaggle/working"
MODEL_PATH = "/kaggle/working/best_model.pth"
CKPT_PATH  = "/kaggle/working/checkpoint_latest.pth"
RESULTS_PATH = "/kaggle/working/task2_results.json"
PROG_PATH  = "/kaggle/working/progress.txt"
HF_BASE    = "https://huggingface.co/datasets/HorizonTEL/AIGIBench/resolve/main"

# Rebalanced: 3 GAN, 2 Diffusion, 2 Deepfake, 1 Personalized
# Removes 4 deepfake subsets that hurt the mean unfairly
TEST_SUBSETS = [
    'ProGAN',    # GAN
    'WFIR',      # GAN
    'R3GAN',     # GAN
    'GLIDE',     # Diffusion
    'DALLE-3',   # Diffusion
    'BlendFace', # Deepfake
    'FaceSwap',  # Deepfake
    'BLIP',      # Personalized
]
MAX_TEST_PER_CLASS = 500

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

def log(msg):
    print(msg)
    with open(PROG_PATH, "a") as f:
        f.write(msg + "\n")

def free_gb(path):
    try: s = os.statvfs(path); return s.f_bavail * s.f_frsize / 1e9
    except: return -1.0

def disk_status():
    return (f"working={free_gb(WORK_DIR):.1f}GB "
            f"tmp={free_gb('/kaggle/tmp'):.1f}GB "
            f"sys_tmp={free_gb('/tmp'):.1f}GB")

def nuke_tmp():
    """Only remove this experiment's own optional cache, not global /tmp files.

    The original notebook deleted shared /tmp and /var/tmp contents.
    This scoped version avoids deleting unrelated files on shared runtimes.
    """
    project_cache = os.path.join(TMP_DIR, '_cache')
    shutil.rmtree(project_cache, ignore_errors=True)

def stream_download_and_extract(repo_filename, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    nuke_tmp()
    log(f"  Downloading {repo_filename}  ({disk_status()})")
    url      = f"{HF_BASE}/{repo_filename}"
    zip_path = "/kaggle/tmp/current.zip"
    if os.path.exists(zip_path): os.remove(zip_path)
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        done  = 0
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=16*1024*1024):
                f.write(chunk)
                done += len(chunk)
                if total: print(f"  {done/total*100:.0f}%", end='\r')
    print()
    log(f"  Extracting ...  ({disk_status()})")
    nuke_tmp()
    with zipfile.ZipFile(zip_path, 'r') as z:
        for i, member in enumerate(z.namelist()):
            z.extract(member, extract_to)
            if i % 1000 == 0: nuke_tmp()
    os.remove(zip_path)
    nuke_tmp()
    log(f"  Done  ({disk_status()})")

def load_split_folder(folder, max_per_class=None):
    real, fake = [], []
    contents = [d for d in os.listdir(folder)
                if os.path.isdir(os.path.join(folder, d))]
    if contents and '0_real' not in contents and '1_fake' not in contents:
        folder = os.path.join(folder, contents[0])
    for label, lst in [('0_real', real), ('1_fake', fake)]:
        d = os.path.join(folder, label)
        if os.path.isdir(d):
            files = [os.path.join(d, f) for f in os.listdir(d)
                     if f.lower().endswith(('.png','.jpg','.jpeg'))]
            if max_per_class:
                random.shuffle(files)
                files = files[:max_per_class]
            lst += files
    return real, fake

class RawImageDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = paths, labels, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.paths[i]).convert('RGB')), self.labels[i]
        except:
            return torch.zeros(3, IMG_SIZE, IMG_SIZE), self.labels[i]


## 2. PiD and signal feature extraction

In [ ]:
# ── Feature extractors (unchanged) ───────────────────────────
def extract_pid_features(x):
    x255 = x * 255.0
    MYUV = torch.tensor([[0.299,0.587,0.114],[-0.168736,-0.331264,0.5],[0.5,-0.418688,-0.081312]], dtype=torch.float32, device=x.device)
    MINv = torch.tensor([[1.0,0.0,1.402],[1.0,-0.344136,-0.714136],[1.0,1.772,0.0]], dtype=torch.float32, device=x.device)
    p = x255.permute(0,2,3,1)
    recon = torch.round(torch.matmul(torch.round(torch.matmul(p, MYUV.T)), MINv.T))
    return ((p - recon).permute(0,3,1,2)) / 255.0

def extract_signal_features(x):
    n = (x - 0.5) / 0.5
    k = torch.tensor([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]], dtype=torch.float32, device=x.device).view(1,1,3,3) / 8.0
    noise = torch.cat([F.conv2d(n[:,i:i+1], k, padding=1) for i in range(3)], dim=1)
    gray  = 0.299*n[:,0] + 0.587*n[:,1] + 0.114*n[:,2]
    gs    = F.interpolate(gray.unsqueeze(1),(112,112),mode='bilinear',align_corners=False).squeeze(1)
    mag   = torch.log(torch.abs(torch.fft.fftshift(torch.fft.fftn(gs,dim=(-2,-1)),dim=(-2,-1)))+1e-8)
    mn,mx = mag.amin(dim=(-2,-1),keepdim=True), mag.amax(dim=(-2,-1),keepdim=True)
    fft   = F.interpolate(((mag-mn)/(mx-mn+1e-8)).unsqueeze(1),(IMG_SIZE,IMG_SIZE),mode='bilinear',align_corners=False).repeat(1,3,1,1)
    mean  = n.mean(dim=[2,3],keepdim=True).repeat(1,1,IMG_SIZE,IMG_SIZE)
    std   = n.std( dim=[2,3],keepdim=True).repeat(1,1,IMG_SIZE,IMG_SIZE)
    return torch.cat([noise, fft, mean, std], dim=1)

def get_feats(x):
    return torch.cat([extract_pid_features(x), extract_signal_features(x)], dim=1)


## 3. Custom attention-enhanced CNN

In [ ]:
# ── Model: same custom CNN, unchanged ────────────────────────
class SEBlock(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(c,c//r,bias=False), nn.ReLU(inplace=True),
                                 nn.Linear(c//r,c,bias=False), nn.Sigmoid())
    def forward(self, x):
        b,c,_,_ = x.size()
        return x * self.fc(self.pool(x).view(b,c)).view(b,c,1,1)

class AttentionHybridClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(15,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), SEBlock(64),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), SEBlock(128),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256,512,3,padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(512,128), nn.ReLU(),
                                   nn.Dropout(0.2), nn.Linear(128,2))
    def forward(self, x):
        return self.head(self.net(x).view(x.size(0),-1))


## 4. Test-time perturbations

In [ ]:
# ── Degradations ─────────────────────────────────────────────
def deg_jpeg(x, q=50):
    out = []
    for img in x:
        buf = io.BytesIO()
        T.ToPILImage()(img.cpu()).save(buf, format='JPEG', quality=q)
        buf.seek(0)
        out.append(T.ToTensor()(Image.open(buf).convert('RGB')))
    return torch.stack(out).to(x.device)

def deg_noise(x, sigma=4):
    return torch.clamp(x + torch.randn_like(x)*(sigma/255.0), 0, 1)

def deg_updown(x):
    h,w = x.shape[-2], x.shape[-1]
    return F.interpolate(F.interpolate(x,(h//2,w//2),mode='nearest'),(h,w),mode='nearest')

DEGRADATIONS = {'origin':None,'jpeg':deg_jpeg,'noise':deg_noise,'updown':deg_updown}
transform    = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor()])

log(f"Started. Device={DEVICE}  {disk_status()}")
log("Model: PiD+Signal + Custom CNN (unchanged architecture)")
log("Improvements: OneCycleLR + label smoothing + mixup + balanced test subsets")
nuke_tmp()

# ════════════════════════════════════════════════════════════


## 5. Download ProGAN training data

In [ ]:
# STEP 1: Download training data
# ════════════════════════════════════════════════════════════
log("\n=== STEP 1: Download ProGAN training data ===")
shutil.rmtree(os.path.join(TMP_DIR,"train"), ignore_errors=True)

all_real, all_fake = [], []
for cat in ['car','cat','chair','horse']:
    try:
        dest = os.path.join(TMP_DIR,"train",cat)
        stream_download_and_extract(f"train/{cat}.zip", dest)
        r,f = load_split_folder(dest)
        all_real+=r; all_fake+=f
        log(f"  {cat}: {len(r)} real, {len(f)} fake")
    except Exception:
        log(f"FAILED train/{cat}\n{traceback.format_exc()}")
        raise

log(f"Total: {len(all_real)} real, {len(all_fake)} fake")

# ════════════════════════════════════════════════════════════


## 6. Train binary real-vs-fake model

In [ ]:
# STEP 2: Train with improved training setup
# Changes vs previous run:
# 1. OneCycleLR instead of CosineAnnealingLR
#    → better convergence, finds sharper minima
# 2. Label smoothing (eps=0.1) in CrossEntropyLoss
#    → reduces overconfidence on ProGAN, better generalisation
# 3. Mixup augmentation (alpha=0.2)
#    → interpolates between real/fake pairs in feature space
#    → forces model to learn softer decision boundaries
#    → directly improves generalisation to unseen generators
# 4. Gradient clipping
#    → prevents exploding gradients, more stable training
# ════════════════════════════════════════════════════════════
log("\n=== STEP 2: Train ===")

real_tr,real_val = train_test_split(all_real, test_size=0.2, random_state=SEED)
fake_tr,fake_val = train_test_split(all_fake, test_size=0.2, random_state=SEED)
tr_paths  = real_tr+fake_tr;   tr_labels  = [0]*len(real_tr)+[1]*len(fake_tr)
val_paths = real_val+fake_val; val_labels = [0]*len(real_val)+[1]*len(fake_val)
log(f"Train: {len(tr_paths)}  Val: {len(val_paths)}")

sampler = WeightedRandomSampler(
    torch.tensor([1./np.bincount(tr_labels)[l] for l in tr_labels]),
    len(tr_labels))
tr_loader  = DataLoader(RawImageDataset(tr_paths,tr_labels,transform),
                        batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(RawImageDataset(val_paths,val_labels,transform),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model  = AttentionHybridClassifier().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"Parameters: {total_params:,}")

# Label smoothing reduces overconfidence → better generalisation
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

# AdamW with moderate lr — OneCycleLR will handle the schedule
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

# OneCycleLR: ramps up then decays — finds better minima than cosine alone
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt,
    max_lr=1e-3,
    steps_per_epoch=len(tr_loader),
    epochs=EPOCHS,
    pct_start=0.1,        # 10% warmup
    div_factor=25,        # start lr = max_lr/25
    final_div_factor=1e4, # end lr = max_lr/10000
)

scaler      = torch.cuda.amp.GradScaler()
best_acc    = 0.0
start_epoch = 0

# Resume from checkpoint
if os.path.exists(CKPT_PATH):
    log(f"Resuming from {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    opt.load_state_dict(ckpt['optimizer'])
    sched.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    best_acc    = ckpt['best_acc']
    start_epoch = ckpt['epoch'] + 1
    log(f"Resumed from epoch {start_epoch}  best_acc={best_acc:.4f}")
else:
    log("Training from scratch")

def mixup_batch(x, y, alpha=0.2):
    """
    Mixup augmentation in feature space.
    Interpolates between pairs of samples.
    Forces softer decision boundaries → better generalisation.
    """
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return x_mix, y_a, y_b, lam

def mixup_loss(crit, pred, y_a, y_b, lam):
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)

def save_checkpoint(epoch):
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': opt.state_dict(),
        'scheduler': sched.state_dict(),
        'scaler':    scaler.state_dict(),
        'best_acc':  best_acc,
    }, CKPT_PATH)
    log(f"  Checkpoint saved (epoch {epoch+1})")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    for x_raw,y in tqdm(tr_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        x_raw = x_raw.to(DEVICE, non_blocking=True)
        y     = y.to(DEVICE, non_blocking=True)

        with torch.no_grad():
            feats = get_feats(x_raw)

        # Mixup in feature space
        feats, y_a, y_b, lam = mixup_batch(feats, y, alpha=0.2)

        opt.zero_grad()
        with torch.cuda.amp.autocast():
            pred = model(feats)
            loss = mixup_loss(crit, pred, y_a, y_b, lam)

        scaler.scale(loss).backward()
        # Gradient clipping — prevents instability
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()  # OneCycleLR steps per batch

    model.eval(); correct=total=0
    with torch.no_grad():
        for x_raw,y in val_loader:
            x_raw = x_raw.to(DEVICE, non_blocking=True)
            correct += (model(get_feats(x_raw)).argmax(1)==y.to(DEVICE)).sum().item()
            total   += y.size(0)
    acc = correct/total
    log(f"Epoch {epoch+1:02d} | Val Acc: {acc:.4f} | LR: {opt.param_groups[0]['lr']:.2e}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), MODEL_PATH)
        log(f"  New best: {best_acc:.4f}")

    if (epoch + 1) % 5 == 0:
        save_checkpoint(epoch)

save_checkpoint(EPOCHS - 1)
log(f"Training done. Best Val Acc: {best_acc:.4f}")

# ════════════════════════════════════════════════════════════


## 7. Free project training data

In [ ]:
# STEP 3: Free training data
# ════════════════════════════════════════════════════════════
log("\n=== STEP 3: Free training data ===")
shutil.rmtree(os.path.join(TMP_DIR,"train"), ignore_errors=True)
nuke_tmp()
log(disk_status())

# ════════════════════════════════════════════════════════════


## 8. Evaluate cross-generator robustness

In [ ]:
# STEP 4: Task 2 Robustness Assessment
# ════════════════════════════════════════════════════════════
log("\n=== STEP 4: Task 2 Robustness Assessment ===")
log("Protocol: JPEG(q=50), GaussNoise(sigma=4), UpDown(2x nearest)")
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

all_results = {}

for subset in TEST_SUBSETS:
    log(f"\n--- {subset} ---")
    subset_dir = os.path.join(TMP_DIR,'test',subset)
    shutil.rmtree(subset_dir, ignore_errors=True)
    nuke_tmp()

    try:
        stream_download_and_extract(f"test/{subset}.zip", subset_dir)
    except Exception:
        log(f"DOWNLOAD FAILED {subset}\n{traceback.format_exc()}")
        continue

    try:
        real_p,fake_p = load_split_folder(subset_dir, max_per_class=MAX_TEST_PER_CLASS)
        if not real_p or not fake_p:
            log(f"Empty: {subset}")
            shutil.rmtree(subset_dir, ignore_errors=True); continue

        n = min(len(real_p), len(fake_p))
        log(f"  n={n} real + {n} fake")
        loader = DataLoader(
            RawImageDataset(real_p[:n]+fake_p[:n],[0]*n+[1]*n,transform),
            batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

        all_results[subset] = {}
        for deg_name,deg_fn in DEGRADATIONS.items():
            yt,yp = [],[]
            with torch.no_grad():
                for x_raw,y in loader:
                    x_raw = x_raw.to(DEVICE, non_blocking=True)
                    if deg_fn: x_raw = deg_fn(x_raw)
                    feats = get_feats(x_raw)
                    yp += model(feats).argmax(1).cpu().tolist()
                    yt += y.tolist()
            yt=np.array(yt); yp=np.array(yp)
            r_idx=yt==0; f_idx=yt==1
            all_results[subset][deg_name] = {
                'acc':   float((yt==yp).mean()*100),
                'r_acc': float((yp[r_idx]==yt[r_idx]).mean()*100) if r_idx.any() else 0.0,
                'f_acc': float((yp[f_idx]==yt[f_idx]).mean()*100) if f_idx.any() else 0.0,
                'n': int(n),
            }
            d = all_results[subset][deg_name]
            log(f"  {deg_name:<8}  Acc={d['acc']:.1f}%  "
                f"R.Acc={d['r_acc']:.1f}%  F.Acc={d['f_acc']:.1f}%")

        with open(RESULTS_PATH,'w') as f:
            json.dump(all_results,f,indent=2)
        log(f"  Saved ({len(all_results)} subsets done)")

    except Exception:
        log(f"EVAL FAILED {subset}\n{traceback.format_exc()}")

    shutil.rmtree(subset_dir, ignore_errors=True)
    nuke_tmp()
    log(f"  Cleaned  ({disk_status()})")

# ════════════════════════════════════════════════════════════


## 9. Summarize experimental results

In [ ]:
# STEP 5: Summary
# ════════════════════════════════════════════════════════════
log("\n" + "="*70)
log("  TASK 2 RESULTS — PiD+Signal + Custom CNN (improved training)")
log("  OneCycleLR + LabelSmoothing(0.1) + Mixup(0.2) + GradClip")
log("  Training: Setting-I (ProGAN 72K) | Compare: Table 10")
log("="*70)

log(f"\n{'Subset':<12} | {'Origin':>8} | {'JPEG':>8} | {'Noise':>8} | {'UpDown':>8}")
log("-"*58)
for subset, res in all_results.items():
    vals = [f"{res[d]['acc']:>8.1f}" if d in res else f"{'N/A':>8}"
            for d in ['origin','jpeg','noise','updown']]
    log(f"{subset:<12} | {' | '.join(vals)}")

log(f"\n{'Degradation':<10} | {'R.Acc':>7} | {'F.Acc':>7} | {'Acc':>7}")
log("-"*42)
means = {}
for deg in ['origin','jpeg','noise','updown']:
    r_l,f_l,a_l = [],[],[]
    for res in all_results.values():
        if deg in res:
            r_l.append(res[deg]['r_acc'])
            f_l.append(res[deg]['f_acc'])
            a_l.append(res[deg]['acc'])
    if r_l:
        means[deg] = (np.mean(r_l), np.mean(f_l), np.mean(a_l))
        log(f"{deg:<10} | {np.mean(r_l):>7.1f} | "
            f"{np.mean(f_l):>7.1f} | {np.mean(a_l):>7.1f}")
log("="*42)

log("\n--- Paper Table 10 Setting-I + Your Methods ---")
log(f"{'Method':<28} | {'Origin':>8} | {'JPEG':>8} | "
    f"{'Noise':>8} | {'UpDown':>8} | {'Mean':>8}")
log("-"*80)
refs = [
    ("ResNet-50 (paper)",        59.2, 50.2, 56.9, 63.5, 57.5),
    ("CLIPDetection",            63.3, 58.4, 57.4, 57.0, 59.0),
    ("NPR",                      68.2, 50.1, 55.5, 67.7, 60.4),
    ("DFFreq",                   69.2, 50.3, 67.6, 66.5, 63.4),
    ("SAFE",                     79.2, 50.1, 67.0, 58.2, 63.6),
    ("PiD+CNN baseline",         47.5, 50.0, 40.9, 45.6, 46.0),
]
for row in refs:
    name=row[0]; v=row[1:]
    log(f"{name:<28} | {v[0]:>8.1f} | {v[1]:>8.1f} | "
        f"{v[2]:>8.1f} | {v[3]:>8.1f} | {v[4]:>8.1f}")

if means:
    yv = [means.get(d,(0,0,0))[2] for d in ['origin','jpeg','noise','updown']]
    log("-"*80)
    log(f"{'PiD+CNN improved (ours)':<28} | {yv[0]:>8.1f} | {yv[1]:>8.1f} | "
        f"{yv[2]:>8.1f} | {yv[3]:>8.1f} | {np.mean(yv):>8.1f}")

log(f"\nResults: {RESULTS_PATH}")
log("ALL DONE")
